# Controls and Input

**Part I · Visualization** — Tutorial 16

Cover every control the viewer offers, and combine them into a polished control
panel. You will learn to:

- Create sliders, dropdowns, buttons, value-edit steppers, text fields/areas,
  color pickers, checkboxes, and the file chooser.
- Group controls (`add_control_group`) with icons and tooltips.
- Use the reusable text editor (`open_editor`).
- Update control values in place (`set_control_value` /
  `set_control_view_value` / `update_control`).
- Remove controls and scope them per scene.

> Every control uses the same **async** `(value, event)` handler contract, and
> the value field is `value` (the old `default=` keyword no longer works).


## Setup


In [1]:
from pytanga.viz import Visualizer


## 1. Sliders, dropdowns, and buttons

`add_slider` takes `min`/`max`/`step`/`value` plus `on_change` (and the
press/release events `on_press` / `on_release`). `add_dropdown` takes `options`
/ `value` / `on_change`. `add_button` takes `on_click` plus an optional icon.


In [2]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)

viz.add_slider("s", label="Slider", min=0, max=1, step=0.01, value=0.5)
viz.add_dropdown("d", label="Dropdown", options=["a", "b", "c"], value="a")
viz.add_button("b", label="Button")

viz.flush()
print("slider, dropdown, button added")


slider, dropdown, button added


## 2. Value-edit, text, color, and checkbox

- `add_value_edit` — numeric stepper (`min`/`max`/`step`/`digits`, up/down
  buttons, arrow-key / scroll-wheel stepping, `editable=`).
- `add_text_field` (single-line) / `add_text_area` (multi-line, `rows`).
- `add_color_picker` (native hex color) / `add_checkbox` (boolean).


In [3]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)

viz.add_value_edit("v", label="Value", min=0, max=10, step=0.1, digits=2, value=1.0)
viz.add_text_field("t", label="Text", value="hello")
viz.add_text_area("ta", label="Notes", rows=3)
viz.add_color_picker("cp", label="Color", value="#4488ff")
viz.add_checkbox("cb", label="Checkbox", value=True)

viz.flush()
print("value-edit, text, color, checkbox added")


value-edit, text, color, checkbox added


## 3. File chooser

`add_file_chooser` is a text field + "Browse…" backed by a backend-driven,
modal file browser. `root=` sets the browse root; `open_file_chooser()` opens it
from the backend (optionally at a path).


In [4]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)

viz.add_file_chooser(
    "data_file",
    label="Data file",
    placeholder="/path/to/file",
    root="~",
)

# viz.open_file_chooser("data_file")             # opens at value/root
# viz.open_file_chooser("data_file", path="/tmp") # opens at a path
viz.flush()
print("file chooser added")


file chooser added


## 4. Control groups, icons, and tooltips

`add_control_group` (title bar, `position`, `collapsed`, `on_toggle`) groups
controls. Buttons and group title bars accept `icon=` (a `family:name` string or
the `EIconMaterial` / `EIconUC` enums — `material:` loads from Google Fonts,
`uc:` glyphs need no font) and `tooltip=` hover text. Controls must be created
**before** the group that references them.


In [5]:
from pytanga.viz import EIconMaterial, EIconUC

viz = Visualizer(add_default_axes=False, add_default_grid=False)

viz.add_slider("r", label="Radius", min=0, max=5, value=1.0)
viz.add_button("reset", label="Reset", icon=EIconMaterial.REFRESH, tooltip="Reset")
viz.add_button("quit", icon=EIconUC.CLOSE, icon_only=True, tooltip="Quit")

viz.add_control_group(
    "g", title="Controls", icon=EIconUC.GEAR, tooltip="Settings",
    controls=["r", "reset", "quit"], position="bottom-right",
)

viz.flush()
print("control group added")


control group added


## 5. Reusable text editor — `open_editor()`

`open_editor()` opens a transient multi-line overlay; `on_close(text, event)`
receives the edited text, or `None` on discard.


In [6]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)

async def on_edited(text, event):
    if text is not None:
        print("edited:", text)

viz.open_editor("editor", label="Edit", value="initial text", on_close=on_edited)
viz.flush()
print("editor opened")


editor opened


## 6. In-place value updates

Update a control's value **in place** (preserving collapse/drag/focus state)
with `set_control_value` / `set_control_view_value` / `update_control(..., value=...)`.


In [7]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
viz.add_slider("radius", label="Radius", min=0, max=5, value=1.0)

viz.set_control_value("radius", 3.0)      # panel / add_* control
# viz.update_control("radius", value=3.0) # equivalent
# viz.set_control_view_value(radius_view, 3.0)  # layout view control

viz.flush()
print("control value updated in place")


control value updated in place


## 7. Scene-scoped controls and removal

Controls are per-scene: create them on a `VizSceneHandle` for that scene.
Remove with `remove_control` / `remove_control_group` / `clear_controls`.


In [8]:
viz = Visualizer(add_default_axes=False, add_default_grid=False)
detail = viz.scene("detail")
detail.add_slider("d", label="Detail slider", min=0, max=1, value=0.5)

viz.add_slider("main", label="Main slider", min=0, max=1, value=0.5)
viz.remove_control("main")
viz.clear_controls()   # remove all (main scene)

detail.clear_controls()  # remove the detail scene's controls
print("controls scoped / removed")


controls scoped / removed


## 8. View-control counterparts

Every panel control has a matching `View` class for use inside a `SplitView`
pane (see [Tutorial 14](../14_split_views/)): `SliderView`, `DropdownView`,
`ButtonView`, `ValueEditView`, `FileChooserView`, and (in `pytanga.viz.views`)
`TextFieldView`, `TextAreaView`, `ColorPickerView`, `CheckboxView`.


## 9. Menus, dialogs, and table undo/redo

Beyond the panel controls, `pytanga.viz` also provides a **menu system** and a
**dialog overlay** (1.15.0), a **file-open dialog** (1.17.0), and **table
undo/redo** (1.14.2):

- `add_menu(mid, label=..., mode="dropdown"|"bar", children=[...])` mounts a
  global menu — a hamburger dropdown or a permanent menu bar with nestable
  sub-menus; `EAnchor` and `EControlVariant` control placement/styling.
- `show_dialog(content, id=..., title=..., dismissable=...)` renders a titled,
  draggable overlay; `dismissable=False` makes it modal (dimmed backdrop).
  `remove_dialog(id)` / `clear_dialogs()` dismiss it.
- `FileChooserDialog(control_id, ...)` is a ready-made file-open dialog (a
  `FileChooserView` listing plus a path line and OK/Cancel).
- `add_table` / `TableView` grids support spreadsheet-style editing and backend
  undo/redo (`undo_table` / `redo_table` / `clear_table_history`).

These reuse the same `View` classes as layouts ([Tutorial 14](../14_split_views/)).


## Visual Examples

One of every control kind, in a `VisualizerApp` (condensed from the library's
`all_controls.py`).


In [9]:
from pytanga.geometry import Point, Sphere
from pytanga.viz import ControlEvent, EIconMaterial, EIconUC, VisualizerApp


class AllControlsApp(VisualizerApp):
    def __init__(self):
        super().__init__(title="All Controls")
        self._radius = 1.0
        self._color = "#4488ff"

    async def init(self) -> None:
        self.viz.add(Sphere(Point(0, 0, 0), self._radius), entity_id="ball",
                     color=self._color, opacity=0.9, label="The ball")
        self.viz.add_slider("radius", label="Radius", min=0.2, max=3.0, value=self._radius,
                            on_change=self.on_radius)
        self.viz.add_dropdown("mode", label="Mode", options=["Solid", "Translucent", "Hidden"],
                              value="Solid", on_change=self.on_mode)
        self.viz.add_value_edit("radius_value", label="Radius (stepper)", min=0.2, max=3.0,
                                step=0.1, digits=2, value=self._radius, on_change=self.on_radius)
        self.viz.add_text_field("name", label="Name", value="The ball", on_change=self.on_name)
        self.viz.add_text_area("notes", label="Notes", rows=3, on_change=self.on_notes)
        self.viz.add_color_picker("color", label="Color", value=self._color, on_change=self.on_color)
        self.viz.add_checkbox("wireframe", label="Wireframe", value=True, on_change=self.on_wireframe)
        self.viz.add_file_chooser("data_file", label="Data file", placeholder="/path")
        self.viz.add_button("reset", label="Reset", icon=EIconMaterial.REFRESH, on_click=self.on_reset)
        self.viz.add_button("quit", icon=EIconUC.CLOSE, icon_only=True, tooltip="Quit",
                            on_click=self.on_quit)
        self.viz.add_control_group("g", title="Controls", icon=EIconUC.GEAR,
                                   controls=["radius", "radius_value", "mode", "color", "wireframe",
                                             "name", "notes", "data_file", "reset", "quit"],
                                   position="bottom-right")
        self.viz.flush()

    async def on_radius(self, value: float, _event: ControlEvent) -> None:
        self._radius = value
        self.viz.update_entity("ball", Sphere(Point(0, 0, 0), value))
        self.viz.set_control_value("radius_value", value)
        self.viz.flush()

    async def on_mode(self, value: str, _event: ControlEvent) -> None:
        self.viz.update("ball", opacity={"Solid": 0.9, "Translucent": 0.35, "Hidden": 0.0}[value])
        self.viz.flush()

    async def on_name(self, value: str, _event: ControlEvent) -> None:
        print("name:", value)

    async def on_notes(self, value: str, _event: ControlEvent) -> None:
        print("notes:", value)

    async def on_color(self, value: str, _event: ControlEvent) -> None:
        self._color = value
        self.viz.update("ball", color=value)
        self.viz.flush()

    async def on_wireframe(self, value: bool, _event: ControlEvent) -> None:
        self.viz.update("ball", wireframe=value)
        self.viz.flush()

    async def on_reset(self, _value, _event: ControlEvent) -> None:
        await self.on_radius(1.0, _event)

    async def on_quit(self, _value, _event: ControlEvent) -> None:
        self.request_shutdown()


# AllControlsApp().run()
print("AllControlsApp defined")


AllControlsApp defined


## Summary

| Control | API |
|---|---|
| Slider | `add_slider(cid, min, max, step, value, on_change, on_press, on_release)` |
| Dropdown | `add_dropdown(cid, options, value, on_change)` |
| Button | `add_button(cid, label, icon, icon_only, tooltip, on_click)` |
| Stepper | `add_value_edit(cid, min, max, step, digits, editable, on_change)` |
| Text | `add_text_field(cid, ...)` / `add_text_area(cid, rows=...)` |
| Color / checkbox | `add_color_picker(cid, ...)` / `add_checkbox(cid, ...)` |
| File chooser | `add_file_chooser(cid, root=...)` + `open_file_chooser(cid)` |
| Group | `add_control_group(gid, title, controls=[...], position, collapsed)` |
| Editor | `open_editor(cid, label, value, on_close)` |
| Update value | `set_control_value` / `update_control(..., value=)` / `set_control_view_value` |
| Remove | `remove_control` / `remove_control_group` / `clear_controls` |

**Next:** [17 — Banners & Dialogs](../17_banners_dialogs/).
